# Cross-Series Comparison — Spain · Global · Europe

**Central research question**: Do foundation time series models improve inflation forecasting vs classical statistical models? Do MCP/institutional signals add value, and how does this depend on the data context?

| Series | Variable | Test period | Best baseline | C1 Signals |
|--------|----------|-------------|----------------|------------|
| **Spain**  | Spain CPI (INE) | 2021–2024 | ARIMA/SARIMA | MCP=CPI+GDELT (from 2021) |
| **Global** | Global CPI | 2021–2024 | ARIMA | Inst: Fed Funds, EPU, Brent |
| **Europe** | HICP Eurozone | 2021–2024 | SARIMA | MCP=ECB/GDELT + DFR, ESI, TTF |

**Primary metric**: MASE (normalized by naive lag-12 on 2002–2020) — allows comparing the three series on the same scale despite different units.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import warnings
warnings.filterwarnings('ignore')

ROOT    = Path('..').resolve()
RESULTS = ROOT / '08_results'
HORIZONS = [1, 3, 6, 12]

# ══ SPAIN ════════════════════════════════════════════════════════
spain_raw = json.load(open(RESULTS / 'metrics_summary_final.json'))
for fname, key in [
    ('timesfm_C1_inst_metrics.json',  'timesfm_C1_inst'),
    ('timegpt_C1_inst_metrics.json',  'timegpt_C1_inst'),
    ('chronos2_C1_inst_metrics.json', 'chronos2_C1_inst'),
]:
    p = RESULTS / fname
    if p.exists():
        d = json.load(open(p))
        spain_raw[key] = d[key]

# ══ GLOBAL ════════════════════════════════════════════════════════
global_raw = {}
for src in ['rolling_metrics_global.json', 'rolling_metrics_C1_inst_global.json',
            'deep_rolling_metrics_global.json']:
    global_raw.update(json.load(open(RESULTS / src)))
for name in ['chronos2_C1_inst_global', 'timesfm_C1_inst_global', 'timegpt_C1_inst_global']:
    p = RESULTS / f'{name}_metrics.json'
    if p.exists():
        d = json.load(open(p))
        global_raw[name] = d[name]

# ══ EUROPE ════════════════════════════════════════════════════════
europe_raw = {}
for src in ['rolling_metrics_europe.json', 'deep_rolling_metrics_europe.json']:
    d = json.load(open(RESULTS / src))
    for k, v in d.items():
        europe_raw[f'{k}_europe'] = v
for name in [
    'chronos2_C0_europe', 'chronos2_C1_inst_europe', 'chronos2_C1_mcp_europe', 'chronos2_C1_full_europe',
    'timesfm_C0_europe',  'timesfm_C1_inst_europe',  'timesfm_C1_mcp_europe',  'timesfm_C1_full_europe',
    'timegpt_C0_europe',  'timegpt_C1_inst_europe',  'timegpt_C1_mcp_europe',  'timegpt_C1_full_europe',
]:
    p = RESULTS / f'{name}_metrics.json'
    if p.exists():
        d = json.load(open(p))
        europe_raw[name] = d[name]

# ── helpers ────────────────────────────────────────────────────────
def mae(d, h):  return d.get(f'h{h}', {}).get('MAE')
def mase(d, h): return d.get(f'h{h}', {}).get('MASE')

MASE_SCALE = {
    'spain':  spain_raw['naive']['h1']['MAE']  / spain_raw['naive']['h1']['MASE'],
    'global': global_raw['naive']['h1']['MAE'] / global_raw['naive']['h1']['MASE'],
    'europe': europe_raw['naive_europe']['h1']['MAE'] / europe_raw['naive_europe']['h1']['MASE'],
}
SCOLS = {'spain': '#e6550d', 'global': '#31a354', 'europe': '#3182bd'}

print(f'Models: Spain={len(spain_raw)}, Global={len(global_raw)}, Europe={len(europe_raw)}')
print('MASE scale by series:')
for s, v in MASE_SCALE.items():
    print(f'  {s:7s}: {v:.4f} pp')

---
## 1 · Forecast Difficulty — MASE by Series

The MASE of the best model indicates how predictable each series is at each horizon.

In [ ]:
BEST = {
    'spain': {'data': spain_raw,  'stat': 'arima',        'found': 'timesfm_C1_inst', 'color': '#e6550d'},
    'global': {'data': global_raw, 'stat': 'arima',        'found': 'chronos2_C1_inst_global', 'color': '#31a354'},
    'europe': {'data': europe_raw, 'stat': 'sarima_europe','found': 'timesfm_C1_full_europe',  'color': '#3182bd'},
}
LABELS = {'spain': 'Spain (CPI)', 'global': 'Global (CPI)', 'europe': 'Europe (HICP)'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
x = np.arange(len(HORIZONS))

# Left: MASE stat (dotted) vs foundation (solid)
ax = axes[0]
for sid, cfg in BEST.items():
    d = cfg['data']
    sv = [mase(d.get(cfg['stat'],  {}), h) for h in HORIZONS]
    fv = [mase(d.get(cfg['found'], {}), h) for h in HORIZONS]
    ax.plot(x, sv, 's:', color=cfg['color'], lw=1.5, ms=7, alpha=0.65, label=f'{LABELS[sid]} — stat')
    ax.plot(x, fv, 'o-', color=cfg['color'], lw=2.5, ms=8, label=f'{LABELS[sid]} — found. best')
ax.axhline(1, color='black', lw=1.2, ls='--', alpha=0.4, label='MASE=1 (≡ Naive)')
ax.set_xticks(x); ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=11)
ax.set_ylabel('MASE', fontsize=10)
ax.set_title('MASE — Best statistical (dotted) vs\nBest foundation (solid)', fontsize=10, fontweight='bold')
ax.legend(fontsize=7.5, loc='upper left', framealpha=0.85)
ax.grid(alpha=0.25)

# Right: MASE of naive (intrinsic difficulty)
ax2 = axes[1]
for sid, cfg in BEST.items():
    d = cfg['data']
    nk = 'naive_europe' if sid == 'europe' else 'naive'
    vals = [mase(d.get(nk, {}), h) for h in HORIZONS]
    ax2.plot(x, vals, 's-', color=cfg['color'], lw=2.2, ms=8, label=LABELS[sid])
ax2.set_xticks(x); ax2.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=11)
ax2.set_ylabel('MASE of naive lag-12', fontsize=10)
ax2.set_title('Naive MASE — intrinsic difficulty\n(how much room for improvement)', fontsize=10, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.25)

fig.suptitle('Forecast Difficulty by Series', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_comp1_difficulty.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2 · Foundation Models vs Best Statistical Model

Can foundation models beat the best statistical baseline for each series?

In [ ]:
FOUNDATION_BEST = {
    'spain': {'data': spain_raw,  'stat_key': 'arima',        'stat_lbl': 'ARIMA',
               'found_key': 'timesfm_C1_inst',         'found_lbl': 'TimesFM C1_inst',    'color': '#e6550d'},
    'global': {'data': global_raw, 'stat_key': 'arima',        'stat_lbl': 'ARIMA',
               'found_key': 'chronos2_C1_inst_global',  'found_lbl': 'Chronos-2 C1_inst ★','color': '#31a354'},
    'europe': {'data': europe_raw, 'stat_key': 'sarima_europe','stat_lbl': 'SARIMA',
               'found_key': 'timesfm_C1_full_europe',   'found_lbl': 'TimesFM C1_full ★★', 'color': '#3182bd'},
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
x = np.arange(len(HORIZONS))
for ax, (sid, cfg) in zip(axes, FOUNDATION_BEST.items()):
    d = cfg['data']
    stat  = [mae(d.get(cfg['stat_key'],  {}), h) for h in HORIZONS]
    found = [mae(d.get(cfg['found_key'], {}), h) for h in HORIZONS]
    ax.plot(x, stat,  'o-', color='#444', lw=2.5, ms=8, label=f"Best stat.: {cfg['stat_lbl']}", zorder=5)
    ax.plot(x, found, 's-', color=cfg['color'], lw=2.5, ms=8, label=f"Foundation: {cfg['found_lbl']}", zorder=5)
    for i in range(len(HORIZONS)-1):
        col = '#90ee90' if found[i] < stat[i] else '#ffcccc'
        ax.fill_between([x[i],x[i+1]], [stat[i],stat[i+1]], [found[i],found[i+1]], alpha=0.22, color=col)
    for i, h in enumerate(HORIZONS):
        if stat[i] and found[i]:
            delta = (found[i] - stat[i]) / stat[i] * 100
            ypos  = max(stat[i], found[i]) * 1.05
            ax.text(i, ypos, f'{delta:+.0f}%', ha='center', va='bottom', fontsize=9,
                    color='#006600' if delta < 0 else '#880000', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=10)
    ax.set_title(sid.upper(), fontsize=12, fontweight='bold', color=cfg['color'])
    ax.set_xlabel('Horizon', fontsize=9); ax.set_ylabel('MAE', fontsize=9)
    ax.legend(fontsize=8.5, loc='upper left'); ax.grid(alpha=0.22)

fig.suptitle('Foundation vs Statistical by Series\nGreen = foundation wins · Red = statistical wins',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_comp2_foundation_vs_stat.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 · Family comparison by series — MASE at h=1 and h=12

In [ ]:
FAMILIES = {
    'Chronos-2': {
        'spain': (spain_raw,  'chronos2_C0'),
        'global': (global_raw, 'chronos2_C1_inst_global'),
        'europe': (europe_raw, 'chronos2_C1_inst_europe'),
    },
    'TimesFM': {
        'spain': (spain_raw,  'timesfm_C1_inst'),
        'global': (global_raw, 'timesfm_C1_inst_global'),
        'europe': (europe_raw, 'timesfm_C1_full_europe'),
    },
    'TimeGPT': {
        'spain': (spain_raw,  'timegpt_C0'),
        'global': (global_raw, 'timegpt_C1_inst_global'),
        'europe': (europe_raw, 'timegpt_C0_europe'),
    },
}
STAT_REF = {
    'spain': (spain_raw,  'arima'),
    'global': (global_raw, 'arima'),
    'europe': (europe_raw, 'sarima_europe'),
}
fam_colors   = {'Chronos-2': '#52a852', 'TimesFM': '#9467bd', 'TimeGPT': '#ff7f0e'}
series_list  = ['spain', 'global', 'europe']
series_lbls  = ['Spain', 'Global', 'Europe']

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, h in zip(axes, [1, 12]):
    xp = np.arange(len(series_list))
    w  = 0.25
    for i, (fname, fdata) in enumerate(FAMILIES.items()):
        vals = []
        for sid in series_list:
            d, k = fdata[sid]
            v = mase(d.get(k, {}), h)
            vals.append(v if v else 0)
        bars = ax.bar(xp + (i-1)*w, vals, w, label=fname,
                      color=fam_colors[fname], alpha=0.85, edgecolor='white')
        for bar, v in zip(bars, vals):
            if v:
                ax.text(bar.get_x()+bar.get_width()/2, v+0.02,
                        f'{v:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
    # Statistical reference by series
    for j, sid in enumerate(series_list):
        d, k = STAT_REF[sid]
        v = mase(d.get(k, {}), h)
        if v:
            ax.hlines(v, xp[j]-0.38, xp[j]+0.38, colors=SCOLS[sid], lw=2.0, ls='--', alpha=0.65)
    ax.axhline(1, color='black', lw=1.2, ls=':', alpha=0.5, label='MASE=1 (Naive)')
    ax.set_xticks(xp); ax.set_xticklabels(series_lbls, fontsize=11)
    ax.set_ylabel('MASE', fontsize=10)
    ax.set_title(f'Foundation Families — h={h}\n(line = best statistical model for the series)',
                 fontsize=10, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right'); ax.grid(axis='y', alpha=0.22)
fig.suptitle('MASE by family and series — h=1 and h=12', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_comp3_families.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 · Value of C1 signals by series and family

How much do institutional/MCP signals add (or subtract) relative to C0 (or ARIMA for Global)?

In [ ]:
C1_ROWS = [
    # label                          d_ref       k_ref                d_c1       k_c1
    ('Spain — Chronos-2 C1-MCP',   spain_raw, 'chronos2_C0',       spain_raw, 'chronos2_C1'),
    ('Spain — TimesFM  C1-MCP',    spain_raw, 'timesfm_C0',        spain_raw, 'timesfm_C1'),
    ('Spain — TimesFM  C1-inst',   spain_raw, 'timesfm_C0',        spain_raw, 'timesfm_C1_inst'),
    ('Global — Chronos-2 C1-inst',  global_raw,'arima',              global_raw,'chronos2_C1_inst_global'),
    ('Global — TimesFM  C1-inst',   global_raw,'arima',              global_raw,'timesfm_C1_inst_global'),
    ('Europe — Chronos-2 C1-inst',  europe_raw,'chronos2_C0_europe', europe_raw,'chronos2_C1_inst_europe'),
    ('Europe — TimesFM  C1-inst',   europe_raw,'timesfm_C0_europe',  europe_raw,'timesfm_C1_inst_europe'),
    ('Europe — TimesFM  C1-full ★★',europe_raw,'timesfm_C0_europe',  europe_raw,'timesfm_C1_full_europe'),
]

hm = np.full((len(C1_ROWS), 4), np.nan)
for i, (_, d0, k0, d1, k1) in enumerate(C1_ROWS):
    for j, h in enumerate(HORIZONS):
        m0v = mae(d0.get(k0, {}), h)
        m1v = mae(d1.get(k1, {}), h)
        if m0v and m1v:
            hm[i, j] = (m1v - m0v) / m0v * 100

fig, ax = plt.subplots(figsize=(9, 7.5))
norm = TwoSlopeNorm(vmin=-30, vcenter=0, vmax=100)
im = ax.imshow(hm, cmap='RdYlGn_r', norm=norm, aspect='auto')
ax.set_xticks(range(4)); ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=11)
ax.set_yticks(range(len(C1_ROWS))); ax.set_yticklabels([r[0] for r in C1_ROWS], fontsize=9.5)
ax.set_title('Relative Δ MAE: C1 vs reference × 100%\n'
             'Green = C1 improves  |  Red = C1 worsens', fontsize=11, fontweight='bold')
for i in range(len(C1_ROWS)):
    for j in range(4):
        v = hm[i, j]
        if not np.isnan(v):
            ct = 'white' if abs(v) > 35 else 'black'
            ax.text(j, i, f'{v:+.1f}%', ha='center', va='center',
                    fontsize=9.5, color=ct, fontweight='bold')
ax.axhline(2.5, color='white', lw=2.5)
ax.axhline(4.5, color='white', lw=2.5)
plt.colorbar(im, ax=ax, shrink=0.65, label='Δ MAE (%)')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_comp4_c1_effect.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5 · Main Figure — For the Thesis Document

2×3 panel summarizing the full experiment.

In [ ]:
matplotlib.rcParams.update({'font.family': 'DejaVu Sans',
                             'axes.spines.top': False, 'axes.spines.right': False})
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.32)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[0, 2])
ax_d = fig.add_subplot(gs[1, 0])
ax_e = fig.add_subplot(gs[1, 1])
ax_f = fig.add_subplot(gs[1, 2])
xh = np.arange(len(HORIZONS))

# ── A: MAE Profiles — Spain ──────────────────────────────────────
for key, col, lbl, sty, lw in [
    ('sarima',          '#888',    'SARIMA (best stat)',    's:',  1.6),
    ('auto_arima',      '#8B4513', 'AutoARIMA (dyn.)',      'x--', 1.4),
    ('timesfm_C0',      '#ff9e4a', 'TimesFM C0',            'o-',  2.0),
    ('timesfm_C1_inst', '#e6550d', 'TimesFM C1_inst ★',     'P-',  2.4),
    ('chronos2_C0',     '#f4a582', 'Chronos-2 C0',          '^--', 1.6),
]:
    vals = [mae(spain_raw.get(key, {}), h) for h in HORIZONS]
    if all(v for v in vals):
        ax_a.plot(xh, vals, sty, color=col, lw=lw, ms=7, label=lbl)
ax_a.set_xticks(xh); ax_a.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_a.set_ylabel('MAE (pp Spain CPI)', fontsize=9)
ax_a.set_title('(A) Spain — CPI', fontsize=10, fontweight='bold', color='#e6550d')
ax_a.legend(fontsize=7.5, loc='upper left'); ax_a.grid(alpha=0.22)
ax_a.annotate('Foundation does NOT\nbeat statistical', xy=(0.98, 0.96), xycoords='axes fraction',
              fontsize=7.5, va='top', ha='right', color='#880000',
              bbox=dict(boxstyle='round', fc='#ffe0e0', alpha=0.8))

# ── B: MAE Profiles — Global ──────────────────────────────────────
for key, col, lbl, sty, lw in [
    ('arima',                   '#888',    'ARIMA (best stat)',      's:',  1.6),
    ('auto_arima',              '#8B4513', 'AutoARIMA (dyn.)',        'x--', 1.4),
    ('chronos2_C1_inst_global', '#31a354', 'Chronos-2 C1_inst ★★',   'P-',  2.4),
    ('timesfm_C1_inst_global',  '#74c476', 'TimesFM C1_inst',         'o--', 2.0),
    ('timegpt_C1_inst_global',  '#b3e2c8', 'TimeGPT C1_inst',         'v:',  1.6),
]:
    vals = [mae(global_raw.get(key, {}), h) for h in HORIZONS]
    if all(v for v in vals):
        ax_b.plot(xh, vals, sty, color=col, lw=lw, ms=7, label=lbl)
ax_b.set_xticks(xh); ax_b.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_b.set_ylabel('MAE (pp Global CPI)', fontsize=9)
ax_b.set_title('(B) Global — CPI', fontsize=10, fontweight='bold', color='#31a354')
ax_b.legend(fontsize=7.5, loc='upper left'); ax_b.grid(alpha=0.22)
ax_b.annotate('Chronos-2 beats\nARIMA (h≥3)', xy=(0.98, 0.96), xycoords='axes fraction',
              fontsize=7.5, va='top', ha='right', color='#006600',
              bbox=dict(boxstyle='round', fc='#e0f0e0', alpha=0.8))

# ── C: MAE Profiles — Europe ──────────────────────────────────────
for key, col, lbl, sty, lw in [
    ('sarima_europe',          '#888',    'SARIMA (best stat)',   's:',  1.6),
    ('auto_arima_europe',      '#8B4513', 'AutoARIMA (dyn.)',     'x--', 1.4),
    ('timesfm_C0_europe',      '#9ecae1', 'TimesFM C0',           'o-',  2.0),
    ('timesfm_C1_full_europe', '#3182bd', 'TimesFM C1_full ★★',   'P-',  2.4),
    ('chronos2_C0_europe',     '#6baed6', 'Chronos-2 C0',         '^--', 1.6),
]:
    vals = [mae(europe_raw.get(key, {}), h) for h in HORIZONS]
    if all(v for v in vals):
        ax_c.plot(xh, vals, sty, color=col, lw=lw, ms=7, label=lbl)
ax_c.set_xticks(xh); ax_c.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_c.set_ylabel('MAE (HICP points)', fontsize=9)
ax_c.set_title('(C) Europe — HICP', fontsize=10, fontweight='bold', color='#3182bd')
ax_c.legend(fontsize=7.5, loc='upper left'); ax_c.grid(alpha=0.22)
ax_c.annotate('TimesFM C1_full beats\nSARIMA (h≥6)', xy=(0.98, 0.96), xycoords='axes fraction',
              fontsize=7.5, va='top', ha='right', color='#003366',
              bbox=dict(boxstyle='round', fc='#ddeeff', alpha=0.8))

# ── D: MASE h=12 — grouped bars ──────────────────────────────────
groups = {
    'Naive':       [mase(spain_raw.get('naive',{}),12),
                    mase(global_raw.get('naive',{}),12),
                    mase(europe_raw.get('naive_europe',{}),12)],
    'Best\nstat.': [mase(spain_raw.get('arima',{}),12),
                    mase(global_raw.get('arima',{}),12),
                    mase(europe_raw.get('sarima_europe',{}),12)],
    'Best\nfound.':[mase(spain_raw.get('timesfm_C1_inst',{}),12),
                    mase(global_raw.get('chronos2_C1_inst_global',{}),12),
                    mase(europe_raw.get('timesfm_C1_full_europe',{}),12)],
}
xd = np.arange(3)
for i, (k, vals) in enumerate(groups.items()):
    gcol = ['#b0b0b0', '#606060', '#e3a020'][i]
    bars = ax_d.bar(xd + (i-1)*0.25, vals, 0.25, label=k, color=gcol, alpha=0.88, edgecolor='white')
    for bar, v in zip(bars, vals):
        if v:
            ax_d.text(bar.get_x()+bar.get_width()/2, v+0.04, f'{v:.2f}',
                      ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax_d.axhline(1, color='black', lw=1.2, ls='--', alpha=0.5, label='MASE=1 (Naive)')
ax_d.set_xticks(xd); ax_d.set_xticklabels(['Spain','Global','Europe'], fontsize=10)
ax_d.set_ylabel('MASE (h=12)', fontsize=9)
ax_d.set_title('(D) MASE at h=12\nNaive vs Statistical vs Foundation', fontsize=10, fontweight='bold')
ax_d.legend(fontsize=8, loc='upper right'); ax_d.grid(axis='y', alpha=0.22)

# ── E: Compact Δ C1 heatmap ──────────────────────────────────────
HM_E = [
    ('Spain Chronos-2 C1-MCP',  spain_raw,'chronos2_C0',        spain_raw,'chronos2_C1'),
    ('Spain TimesFM  C1-MCP',   spain_raw,'timesfm_C0',         spain_raw,'timesfm_C1'),
    ('Spain TimesFM  C1-inst',  spain_raw,'timesfm_C0',         spain_raw,'timesfm_C1_inst'),
    ('Global Chronos-2 C1-inst', global_raw,'arima',              global_raw,'chronos2_C1_inst_global'),
    ('Global TimesFM  C1-inst',  global_raw,'arima',              global_raw,'timesfm_C1_inst_global'),
    ('Europe TimesFM  C1-inst',  europe_raw,'timesfm_C0_europe',  europe_raw,'timesfm_C1_inst_europe'),
    ('Europe TimesFM  C1-full★★',europe_raw,'timesfm_C0_europe',  europe_raw,'timesfm_C1_full_europe'),
]
hme = np.full((len(HM_E), 4), np.nan)
for i, (_, d0, k0, d1, k1) in enumerate(HM_E):
    for j, h in enumerate(HORIZONS):
        m0v = mae(d0.get(k0,{}),h); m1v = mae(d1.get(k1,{}),h)
        if m0v and m1v: hme[i,j] = (m1v-m0v)/m0v*100
norm_e = TwoSlopeNorm(vmin=-20, vcenter=0, vmax=60)
ime = ax_e.imshow(hme, cmap='RdYlGn_r', norm=norm_e, aspect='auto')
ax_e.set_xticks(range(4)); ax_e.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_e.set_yticks(range(len(HM_E))); ax_e.set_yticklabels([r[0] for r in HM_E], fontsize=8.5)
ax_e.set_title('(E) Δ MAE: C1 vs reference\nGreen=improvement · Red=worse', fontsize=10, fontweight='bold')
for i in range(len(HM_E)):
    for j in range(4):
        v = hme[i,j]
        if not np.isnan(v):
            ct = 'white' if abs(v) > 30 else 'black'
            ax_e.text(j, i, f'{v:+.0f}%', ha='center', va='center', fontsize=8.5, color=ct, fontweight='bold')
ax_e.axhline(2.5, color='white', lw=2.5); ax_e.axhline(4.5, color='white', lw=2.5)
plt.colorbar(ime, ax=ax_e, shrink=0.8, label='Δ MAE (%)')

# ── F: Summary table ──────────────────────────────────────────────
ax_f.axis('off')
rows_f = [
    ['Spain',  'TimesFM C1_inst',   f"{mase(spain_raw.get('timesfm_C1_inst',{}),1):.3f}",
                'ARIMA',             f"{mase(spain_raw.get('arima',{}),12):.3f}",  '~ 0%\n(C1≈C0)'],
    ['Global',  'ARIMA',             f"{mase(global_raw.get('arima',{}),1):.3f}",
                'Chronos-2 C1_inst', f"{mase(global_raw.get('chronos2_C1_inst_global',{}),12):.3f}", '★ −26%'],
    ['Europe',  'SARIMA',            f"{mase(europe_raw.get('sarima_europe',{}),1):.3f}",
                'TimesFM C1_full',   f"{mase(europe_raw.get('timesfm_C1_full_europe',{}),12):.3f}",  '★★ −17%'],
]
headers = ['Series', 'Best h=1', 'MASE h=1', 'Best h=12', 'MASE h=12', 'C1 Value']
tbl = ax_f.table(cellText=rows_f, colLabels=headers, loc='center', cellLoc='center', bbox=[0, 0.1, 1, 0.82])
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#dddddd'); cell.set_text_props(fontweight='bold')
    elif c == 5:
        txt = cell.get_text().get_text()
        cell.set_facecolor('#c8e6c9' if '★' in txt else '#fff9e6')
    cell.set_edgecolor('#aaaaaa')
ax_f.set_title('(F) Comparative Summary', fontsize=10, fontweight='bold', pad=10)

# ── Global ────────────────────────────────────────────────────────
fig.suptitle(
    'Cross-series Comparison: Spain · Global · Europe — Rolling-origin 2021–2024\n'
    'Foundation models vs statistical baseline · Value of C1 signals',
    fontsize=12, fontweight='bold', y=1.01)
legend_patches = [
    mpatches.Patch(color='#e6550d', label='Spain (CPI)'),
    mpatches.Patch(color='#31a354', label='Global (CPI)'),
    mpatches.Patch(color='#3182bd', label='Europe (HICP Eurozone)'),
    mpatches.Patch(color='#8B4513', label='AutoARIMA dynamic (ref.)'),
    mpatches.Patch(color='#e3a020', label='Best Foundation Model'),
    mpatches.Patch(color='#c8e6c9', label='C1 with significant improvement ★'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=6, fontsize=8.2,
           frameon=True, bbox_to_anchor=(0.5, -0.045))
plt.savefig(RESULTS / 'fig_MAIN_comparison.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('=== MAIN FIGURE SAVED: fig_MAIN_comparison.png ===')

---
## 6 · Cross-Series Synthesis — Key Findings

In [ ]:
print('=' * 75)
print('CROSS-SERIES SYNTHESIS — SPAIN / GLOBAL / EUROPE')
print('=' * 75)

H1 = '''
FINDING 1 — Foundation models are competitive but context matters
─────────────────────────────────────────────────────────────────────────
  Spain  : ARIMA (h=12 MAE=1.541) outperforms all foundation models.
           Spanish CPI dynamics are perfectly captured by ARIMA.
  Global : Chronos-2 C1_inst beats ARIMA at h≥3 (−26% at h=12).
           ARIMA remains king at h=1 (MAE=0.191 vs 0.200).
  Europe : TimesFM C1_full beats SARIMA at h≥6 (MAE=1.995 vs 2.411).
           First time any model breaks the MAE<2.0 barrier at h=12.
'''
print(H1)

H2 = '''
FINDING 2 — C1 signals are beneficial ONLY for the right series
─────────────────────────────────────────────────────────────────────────
  Spain  : C1-MCP DEGRADES (+55% TimesFM, +9% Chronos-2).
           Cause: signals available only from 2021, minimal history.
           C1-inst: neutral (−2% at h=12, not significant).
  Global : C1-inst helps Chronos-2 (−26% h=12) and TimesFM (−17% h=12).
  Europe : C1_full (inst+MCP) is the best model (−17% vs C0 at h=12).
           C1-inst alone: modest (−2 to −4%). MCP adds real value.
'''
print(H2)

H3 = '''
FINDING 3 — Family ranking
────────────────────────────────
  Chronos-2 : most robust with global institutional signals.
              Best for Global CPI (MASE h=12 = 0.976 — the only <1.0).
  TimesFM   : most sensitive to MCP signals, clear benefit in Europe.
              Best for HICP Europe (C1_full, MASE h=12 = 1.370).
              Also best for Spain at h=1 (C1_inst, MASE=0.301).
  TimeGPT   : most fragile with extreme signals (carry-forward 2022).
              Consistently worst across all series.
'''
print(H3)

H4 = '''
FINDING 4 — Forecast horizon is key
───────────────────────────────────
  h=1  : ARIMA/SARIMA very hard to beat.
  h=3-6: Foundation starts to be competitive in Global and Europe.
  h=12 : Foundation wins in Global (−26%) and Europe (−17%); ties in Spain.
         For monetary policy decisions (long horizon),
         foundation models with institutional signals are preferable.
'''
print(H4)

# AutoARIMA comparison across series
aa_es = spain_raw.get('auto_arima', {})
aa_gl = global_raw.get('auto_arima', {})
aa_eu = europe_raw.get('auto_arima_europe', {})
ref_es = spain_raw.get('arima', {})
ref_gl = global_raw.get('arima', {})
ref_eu = europe_raw.get('sarima_europe', {})

H5 = 'FINDING 5 — Dynamic AutoARIMA vs fixed statistical (supervisor recommendation)\n'
H5 += '─' * 70 + '\n'
H5 += '  Reselecting SARIMA orders at each rolling origin WORSENS performance vs\n'
H5 += '  the fixed-order model selected once on the full historical sample.\n\n'
for sid, aa_d, ref_d, ref_lbl in [
    ('Spain',  aa_es, ref_es, 'ARIMA'),
    ('Global', aa_gl, ref_gl, 'ARIMA'),
    ('Europe', aa_eu, ref_eu, 'SARIMA'),
]:
    m_aa_1  = mae(aa_d, 1)
    m_ref_1 = mae(ref_d, 1)
    m_aa_12  = mae(aa_d, 12)
    m_ref_12 = mae(ref_d, 12)
    if m_aa_1 and m_ref_1 and m_aa_12 and m_ref_12:
        d1  = (m_aa_1  - m_ref_1)  / m_ref_1  * 100
        d12 = (m_aa_12 - m_ref_12) / m_ref_12 * 100
        H5 += f'  {sid:7s}: h=1  AutoARIMA={m_aa_1:.4f} vs {ref_lbl}={m_ref_1:.4f} ({d1:+.1f}%)\n'
        H5 += f'           h=12 AutoARIMA={m_aa_12:.4f} vs {ref_lbl}={m_ref_12:.4f} ({d12:+.1f}%)\n'
    else:
        H5 += f'  {sid:7s}: data not available\n'
H5 += ('  Cause: auto_arima overfits to the recent subsample of each rolling window;\n'
       '  fixed orders generalize better to the long horizon.\n'
       '  Implication: calibrating once on the full historical sample\n'
       '  outperforms automatic reselection in rolling.\n')
print(H5)

HM = '''
METHODOLOGICAL FINDING — Ridge + StandardScaler
──────────────────────────────────────────────
  Without scaling   : spurious coefficients (EPU std~65 vs diff(HICP) std~0.44)
                      → constant correction +1.11 pp, TimesFM MAE +534% vs C0.
  With StandardScaler: proportional correction (+0.02 stable → +0.58 peak 2022).
  Lesson             : signals of very different scale require prior normalization.
'''
print(HM)
print('=' * 75)